In [19]:
import pandas as pd
import os

def process_sensor_data(input_folder, output_file, name_list, y_list,  m_liste ):
    all_rows = []
    
    # Wir gehen die Listen 'name' und 'y' parallel durch (zip)
    # Das garantiert die richtige Reihenfolge (1.csv, 2.csv, ...)
    for i, label_values in enumerate (y_list):
        
        # Pfad sauber zusammenbauen (funktioniert auf Windows und Linux)
        file_path = os.path.join(input_folder, f"{i+1}.csv")

        if not os.path.exists(file_path):
            print(f"Warnung: Datei {file_path} wurde nicht gefunden. Überspringe...")
            continue
            
        # Datei laden
        df = pd.read_csv(file_path)
        print(f"Verarbeite {file_path} mit {len(df)} Zeilen und Spalten: {df.columns.tolist()}")
        # 1. Unnötige Spalten entfernen
        if 'timestamp' in df.columns:
            df = df.drop(columns=['timestamp'])
            
        # 2. Daten transformieren: 'step' als Index, dann flach ausrollen (unstack)
        df = df.set_index('step')
        unstacked = df.unstack()
        
        # 3. Neue Spaltennamen vergeben (z.B. A_1_front, A_2_left)
        # Die Reihenfolge der Messpunkte bleibt durch das unstack() erhalten
        new_columns = [f"{sensor}_{step.replace('data_recording_', '')}" 
                      for sensor, step in unstacked.index]
        
        # 4. Eine einzelne Zeile für den DataFrame erstellen
        row_df = pd.DataFrame([unstacked.values], columns=new_columns)
        
        # 5. Die y-Labels hinzufügen (y_1, y_2, y_3)
        row_df['x'] = label_values[0]
        row_df['y'] = label_values[1]
        row_df['z'] = label_values[2]
        row_df['source_file'] = f"{name_list[i]}.csv"
        row_df['Objektname'] = m_liste[i][0]
        row_df['Objektkategorie'] = m_liste[i][1]
        row_df['OAufnahme'] = m_liste[i][2]
        row_df['Schwierigkeit'] = m_liste[i][3]
        row_df['Gewicht'] = m_liste[i][4]
        
        all_rows.append(row_df)
    
    # Alle Zeilen zu einem großen DataFrame verbinden
    if all_rows:
        final_df = pd.concat(all_rows, ignore_index=True)
        
        # Speichern
        final_df.to_csv(output_file, index=False)
        print(f"Erfolgreich erstellt! CSV hat {len(final_df)} Zeilen.")
        return final_df
    else:
        print("Keine Daten verarbeitet.")
        return None

In [20]:
def generate_elements():
    elements = []
    
    # 4eck_g_O_V_u_1 bis 10
    elements.extend([f"4eck_g_O_V_u_{i}" for i in range(1, 11)])
    # 4eck Einzelwerte
    elements.extend(['4eck_g_O_V_r_4', '4eck_g_O_V_r_7', '4eck_g_L_V_O_2', '4eck_g_L_V_O_10', 
                     '4eck_g_L_V_O_9', '4eck_g_L_V_O_7', '4eck_g_V_V_O_4', '4eck_g_V_V_O_3', 
                     '4eck_g_V_V_O_5', '4eck_g_V_V_O_7'])
    
    # 3eck & Rechteck (1 bis 7)
    elements.extend([f"3eck_L_U_{i}" for i in range(1, 8)])
    elements.extend([f"3eck_L_L_{i}" for i in range(1, 8)])
    elements.extend([f"Rechteck_L_U_{i}" for i in range(1, 8)])
    elements.extend([f"Rechteck_L_L_{i}" for i in range(1, 8)])
    
    # Rechteck H & V (1 bis 3)
    elements.extend([f"Rechteck_H_L_U_{i}" for i in range(1, 4)])
    elements.extend([f"Rechteck_H_L_L_{i}" for i in range(1, 4)])
    elements.extend([f"Rechteck_V_L_U_{i}" for i in range(1, 4)])
    elements.extend([f"Rechteck_V_L_L_{i}" for i in range(1, 4)])
    
    # Lumi L (1 bis 13)
    elements.extend([f"Lumi_L_U_{i}" for i in range(1, 14)])
    elements.extend([f"Lumi_L_L_{i}" for i in range(1, 14)])
    
    # Lumi H (1 bis 3) & V (1 bis 6)
    elements.extend([f"Lumi_H_L_U_{i}" for i in range(1, 4)])
    elements.extend([f"Lumi_H_L_L_{i}" for i in range(1, 4)])
    elements.extend([f"Lumi_V_L_U_{i}" for i in range(1, 7)])
    elements.extend([f"Lumi_V_L_L_{i}" for i in range(1, 7)])
    
    # Durch (1 bis 6)
    elements.extend([f"Durch_L_U_D_O_{i}" for i in range(1, 7)])
    elements.extend([f"Durch_L_U_D_U_{i}" for i in range(1, 7)])
    
    # 6eck (1 bis 8) & Einzelwerte
    elements.extend([f"6eck_L_U_{i}" for i in range(1, 9)])
    elements.extend([f"6eck_L_L_{i}" for i in range(1, 9)])
    elements.extend(['6eck_H_L_U_1', '6eck_H_L_U_2'])
    
    # Endboss (1 bis 3)
    elements.extend([f"Endboss_L_U_S_0_{i}" for i in range(1, 4)])
    elements.extend([f"Endboss_L_U_S_X_{i}" for i in range(1, 4)])
    elements.extend([f"Endboss_L_L_S_0_{i}" for i in range(1, 4)])
    elements.extend([f"Endboss_L_L_S_Y_{i}" for i in range(1, 4)])

    # Durch (1 bis 6)
    elements.extend([f"Durch_L_L_D_O_{i}" for i in range(1, 7)])
    elements.extend([f"Durch_L_L_D_U_{i}" for i in range(1, 7)])
    
    # Rechteck Lang (1 bis 6)
    elements.extend([f"Rechteck_Lang_L_U_{i}" for i in range(1, 7)])
    elements.extend([f"Rechteck_Lang_L_L_{i}" for i in range(1, 7)])

    return elements

# Aufruf der Funktion erstellt das exakt gleiche Array:
elemente_liste = generate_elements()

In [21]:
def generate_y():
    y = []
    
    # 4eck_g_O_V_u_1 bis 10
    y.extend([[50 , 50 , 50] for i in range(1, 21)])
    
    # 3eck 
    y.extend([[91 , 53.33 , 30] for i in range(1, 8)])
    y.extend([[53.33, 91, 30] for i in range(1, 8)])

    # Rechteck (1 bis 7)
    y.extend([[50 , 31.75 , 25] for i in range(1, 8)])
    y.extend([[31.75 , 50 , 25] for i in range(1, 8)])
    
    # Rechteck H & V (1 bis 3)
    y.extend([[25 , 31.75 , 50] for i in range(1, 4)])
    y.extend([[31.75 , 25 , 50] for i in range(1, 4)])
    y.extend([[50 , 25 , 31.75] for i in range(1, 4)])
    y.extend([[25 , 50, 31.75] for i in range(1, 4)])
    
    # Lumi L (1 bis 13)
    y.extend([[87.5 , 47.5 , 20] for i in range(1, 14)])
    y.extend([[47.5 , 87.5 , 20] for i in range(1, 14)])
    
    # Lumi H (1 bis 3) & V (1 bis 6)
    y.extend([[47.5 ,20 , 87.5] for i in range(1, 4)])
    y.extend([[20 , 47.5 , 87.5] for i in range(1, 4)])
    y.extend([[87.5 , 20 , 47.5] for i in range(1, 7)])
    y.extend([[20 , 87.5 , 47.5] for i in range(1, 7)])
    
    # Durch (1 bis 6)
    y.extend([[47.5 , 32.5 , 17.25] for i in range(1, 7)])
    y.extend([[47.5 , 32.5 , 5.75] for i in range(1, 7)])
    
    # 6eck (1 bis 8) & Einzelwerte
    y.extend([[54.5 , 47.5 , 20] for i in range(1, 9)])
    y.extend([[47.5 , 54.5 , 20] for i in range(1, 9)])
    y.extend([[54.5 , 20 , 47.5] for i in range(1, 3)])

    
    # Endboss (1 bis 3)
    y.extend([[64.61 , 46.35 , 20] for i in range(1, 4)])
    y.extend([[81.39 , 46.35 , 20] for i in range(1, 4)])
    y.extend([[46.35 , 64.61 , 20] for i in range(1, 4)])
    y.extend([[46.35 , 81.39 , 20] for i in range(1, 4)])

    # Durch 2 (1 bis 6)
    y.extend([[32.5 , 47.5 , 17.25] for i in range(1, 7)])
    y.extend([[32.5 , 47.5 , 5.75] for i in range(1, 7)])
    
    # Rechteck Lang (1 bis 7)
    y.extend([[120 , 12.5 , 17.5] for i in range(1, 8)])
    y.extend([[12.5 , 120 , 17.5] for i in range(1, 8)])

    return y

# Aufruf der Funktion erstellt das exakt gleiche Array:
y_liste = generate_y()

In [22]:
def generate_y():
    y = []
    
    # 4eck_g_O_V_u_1 bis 10
    y.extend([[50 , 50 , 50] for i in range(1, 21)])
    
    # 3eck 
    y.extend([[91 , 53.33 , 30] for i in range(1, 8)])
    y.extend([[53.33, 91, 30] for i in range(1, 8)])

    # Rechteck (1 bis 7)
    y.extend([[50 , 31.75 , 25] for i in range(1, 8)])
    y.extend([[31.75 , 50 , 25] for i in range(1, 8)])
    
    # Rechteck H & V (1 bis 3)
    y.extend([[25 , 31.75 , 50] for i in range(1, 4)])
    y.extend([[31.75 , 25 , 50] for i in range(1, 4)])
    y.extend([[50 , 25 , 31.75] for i in range(1, 4)])
    y.extend([[25 , 50, 31.75] for i in range(1, 4)])
    
    # Lumi L (1 bis 13)
    y.extend([[87.5 , 47.5 , 20] for i in range(1, 14)])
    y.extend([[47.5 , 87.5 , 20] for i in range(1, 14)])
    
    # Lumi H (1 bis 3) & V (1 bis 6)
    y.extend([[47.5 ,20 , 87.5] for i in range(1, 4)])
    y.extend([[20 , 47.5 , 87.5] for i in range(1, 4)])
    y.extend([[87.5 , 20 , 47.5] for i in range(1, 7)])
    y.extend([[20 , 87.5 , 47.5] for i in range(1, 7)])
    
    # Durch (1 bis 6)
    y.extend([[47.5 , 32.5 , 17.25] for i in range(1, 7)])
    y.extend([[47.5 , 32.5 , 5.75] for i in range(1, 7)])
    
    # 6eck (1 bis 8) & Einzelwerte
    y.extend([[54.5 , 47.5 , 20] for i in range(1, 9)])
    y.extend([[47.5 , 54.5 , 20] for i in range(1, 9)])
    y.extend([[54.5 , 20 , 47.5] for i in range(1, 3)])

    
    # Endboss (1 bis 3)
    y.extend([[64.61 , 46.35 , 20] for i in range(1, 4)])
    y.extend([[81.39 , 46.35 , 20] for i in range(1, 4)])
    y.extend([[46.35 , 64.61 , 20] for i in range(1, 4)])
    y.extend([[46.35 , 81.39 , 20] for i in range(1, 4)])

    # Durch 2 (1 bis 6)
    y.extend([[32.5 , 47.5 , 17.25] for i in range(1, 7)])
    y.extend([[32.5 , 47.5 , 5.75] for i in range(1, 7)])
    
    # Rechteck Lang (1 bis 6)
    y.extend([[120 , 12.5 , 17.5] for i in range(1, 7)])
    y.extend([[12.5 , 120 , 17.5] for i in range(1, 7)])

    return y

# Aufruf der Funktion erstellt das exakt gleiche Array:
y_liste = generate_y()

In [23]:
def generate_more():
    m = []
    
    # 4eck (1 bis 20)
    m.extend([['4eck' , 1 , 2, 1, 185.5] for i in range(1, 21)])
    
    # 3eck (1 bis 14)
    m.extend([['3eck' , 2 , 2, 1, 443.8] for i in range(1, 15)])

    # Rechteck (1 bis 26)
    m.extend([['Rechteck' , 3 , 2 , 1 , 127.8] for i in range(1, 27)])
    
    # Lumi (1 bis 44)
    m.extend([['Lumi L' , 4 , 2, 2 , 51.7] for i in range(1, 45)])
    
    # Durch (1 bis 12)
    m.extend([['Durchsichtig' , 5 , 2 , 3 , 19.3] for i in range(1, 13)])
    
    # 6eck (1 bis 20) & Einzelwerte
    m.extend([['6eck' , 6 , 2 ,1 , 43.9] for i in range(1, 21)])
    
    # Endboss (1 bis 3)
    m.extend([['Endboss' , 7 , 2 , 3 , 58.1] for i in range(1, 13)])

    # Durch 2 (1 bis 6)
    m.extend([['Durchsichtig' , 5 , 3 , 3 , 19.3] for i in range(1, 13)])

    # Rechteck Lang (1 bis 12)
    m.extend([['RechteckLang' , 8 , 3 , 2 , 71.6] for i in range(1, 13)])

    return m

# Aufruf der Funktion erstellt das exakt gleiche Array:
m_liste = generate_more()

In [24]:
df_final = process_sensor_data('./measured_data/Daten', 'training_data_ready.csv', elemente_liste, y_liste,m_liste)

Verarbeite ./measured_data/Daten\1.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\2.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\3.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\4.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\5.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\6.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\7.csv mit 5 Zeilen und Spalten: ['timestamp', 'step', 'A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
Verarbeite ./measured_data/Daten\8.csv mit 5 Zei